<a href="https://colab.research.google.com/github/kiki286/dream-classifier/blob/main/notebooks/02_feature_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 - Feature Extraction
This notebook performs feature extraction on cleaned dream text using two methods:
1. TF-IDF (Term Frequency-Inverse Document Frequency)
2. Sentence Embeddings (using MiniLM from HuggingFace's `sentence-transformers`)


## Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import save_npz


## Load Cleaned Dream Data
Filter out dreams that are too short to be meaningful.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
file_path = "/content/drive/MyDrive/processed_dreams/cleaned_dreams.csv"

df = pd.read_csv(file_path)
print(f"Total dreams loaded: {len(df)}")

df["text_clean"] = df["text_clean"].fillna("")
df = df[df["text_clean"].str.split().str.len() > 10]
print(f"Dreams after filtering short ones: {len(df)}")


Mounted at /content/drive
Total dreams loaded: 22415
Dreams after filtering short ones: 22019


## TF-IDF Feature Extraction
Now convert the text into a sparse matrix using TF-IDF, which down-weights common words and highlights more unique terms in each dream.

In [ ]:
vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
X_tfidf = vectorizer.fit_transform(df["text_clean"])

print("TF-IDF matrix shape:", X_tfidf.shape)

# Save the matrix
os.makedirs("data/processed", exist_ok=True)
save_npz("data/processed/tfidf_matrix.npz", X_tfidf)

# Optionally: Save feature names
with open("data/processed/tfidf_features.txt", "w") as f:
    for word in vectorizer.get_feature_names_out():
        f.write(word + "\n")


TF-IDF matrix shape: (22019, 1000)


## Step 4: Sentence Embeddings
Now use a pre-trained transformer (MiniLM) to compute a dense semantic embedding of each dream. This allows the model to capture meaning beyond surface word counts.

In [ ]:
!pip install -U sentence-transformers

from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
X_embed = model.encode(df["text_clean"].tolist(), show_progress_bar=True)

print("Embedding shape:", X_embed.shape)

# Save to .npy
np.save("data/processed/embeddings.npy", X_embed)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/689 [00:00<?, ?it/s]

Embedding shape: (22019, 384)


## Summary
Extracted two forms of features from the dream texts:
- **TF-IDF** vectors: 1000-dimensional sparse vectors with interpretable word weights
- **Sentence embeddings**: 384-dimensional dense semantic vectors from a transformer model

These are now ready for dimensionality reduction (e.g. PCA, UMAP) and clustering in the next notebook.